# nb00 — свежая книга событий и честный baseline

**Что делаем.** Soft zero: книга событий пересобрана с нуля (`_build_events.py`,
детект v2 как отправная точка). Здесь: (1) санитарная проверка книги,
(2) распределение глубины кластеров k, (3) **главное** — доходность единственного
входа, который v2 доказала исполнимым: весь размер на первом триггере,
по сетке горизонтов выхода 15м…2880м.

**Чего здесь принципиально НЕТ:** усреднения вниз, расписаний заливки, знания k
при входе. `frac = 1` по построению — класс сайзинг-lookahead багов (урок v2 nb07)
здесь невозможен.

Окна хартии: TRAIN 2024-01..2025-06 · VALID 2025-07..2026-01 · TEST 2026-02..07.

In [1]:
import sys; sys.path.insert(0, '.')
from _lab import *

X = pd.read_parquet('_out/events.parquet')
X['entry'] = pd.to_datetime(X['entry'], utc=True)

def window(t):
    if t < pd.Timestamp('2025-07-01', tz='UTC'): return 'TRAIN'
    if t < pd.Timestamp('2026-02-01', tz='UTC'): return 'VALID'
    return 'TEST'
X['win'] = X.entry.map(window)

print('events total:', len(X), '| symbols:', X.sym.nunique())
print()
t = X.pivot_table(index='stream', columns='win', values='sym', aggfunc='size')[['TRAIN','VALID','TEST']]
t['total'] = t.sum(axis=1)
print('=== события по ногам и окнам ===')
print(t.to_string())

events total: 70858 | symbols: 585

=== события по ногам и окнам ===
win     TRAIN  VALID   TEST  total
stream                            
dump     6388   7522   7108  21018
pump    18933  16630  14277  49840


## 1. Глубина кластера k

k = сколько триггерных баров было в кластере. Записана **только для анализа** —
вход её не использует. Смотрим форму: v2 говорила, что хвост k>=7 у дампа — это
~1% событий, но именно он носил все большие убытки.

In [2]:
def kbucket(k):
    if k <= 1: return '1'
    if k == 2: return '2'
    if k == 3: return '3'
    if k <= 6: return '4-6'
    return '7+'
X['kb'] = X.k.map(kbucket)
order = ['1','2','3','4-6','7+']
for s in ('pump','dump'):
    sub = X[X.stream == s]
    d = sub.groupby('kb').size().reindex(order)
    print(f'{s}: ', {b: f'{n} ({n/len(sub)*100:.1f}%)' for b, n in d.items()})

pump:  {'1': '12102 (24.3%)', '2': '6208 (12.5%)', '3': '4292 (8.6%)', '4-6': '8850 (17.8%)', '7+': '18388 (36.9%)'}
dump:  {'1': '5386 (25.6%)', '2': '2701 (12.9%)', '3': '1827 (8.7%)', '4-6': '3479 (16.6%)', '7+': '7625 (36.3%)'}


## 2. Честный baseline: first-trigger, полный размер, сетка горизонтов

Читаем как: строка = горизонт выхода в минутах, значение = средняя сделка в %
(net издержек, катастроф-стоп 30/20% может сработать раньше).

**Унаследованная гипотеза для проверки (актив №4 хартии):** памп хочет короткий
горизонт (~15-30м), дамп — длинный (720м+). Если свежая книга это не воспроизведёт —
актив снимается.

In [3]:
HZ = [15,30,60,120,240,480,720,1440,2880]
for s in ('pump','dump'):
    sub = X[X.stream == s]
    print(f'=== {s}: mean pnl % по горизонтам и окнам (n TRAIN={len(sub[sub.win=="TRAIN"])}, VALID={len(sub[sub.win=="VALID"])}, TEST={len(sub[sub.win=="TEST"])}) ===')
    rows = []
    for hz in HZ:
        col = f'pnl{hz}'
        r = {'hz': hz}
        for w in ('TRAIN','VALID','TEST'):
            p = sub[sub.win == w][col].dropna()
            r[w] = round(p.mean()*100, 3)
        rows.append(r)
    print(pd.DataFrame(rows).set_index('hz').to_string())
    print()

=== pump: mean pnl % по горизонтам и окнам (n TRAIN=18933, VALID=16630, TEST=14277) ===
      TRAIN  VALID   TEST
hz                       
15   -0.257 -0.593 -0.200
30   -0.196 -0.663 -0.172
60   -0.335 -0.928 -0.132
120  -0.498 -1.161 -0.143
240  -0.679 -1.339 -0.063
480  -1.134 -1.511  0.145
720  -1.388 -1.692 -0.033
1440 -1.690 -1.196 -0.249
2880 -2.137 -2.017 -0.409

=== dump: mean pnl % по горизонтам и окнам (n TRAIN=6388, VALID=7522, TEST=7108) ===
      TRAIN  VALID   TEST
hz                       
15    0.566 -0.720 -0.014
30    1.139 -1.163  0.054
60    0.934 -0.720  0.080
120   1.403 -0.599 -0.036
240   1.761 -0.595 -0.184
480   2.225 -0.089 -0.437
720   2.609  0.440  0.001
1440  3.594 -0.542 -0.046
2880  4.236 -0.435 -0.355



## 3. Не только средняя: медиана, win-rate, хвост q10

Урок HOW_WE_WORK: win-rate — ловушка, среднюю без хвоста читать нельзя.
Берём по два «интересных» горизонта на ногу по результатам таблицы выше.

In [4]:
def stats(p):
    p = p.dropna()
    return dict(n=len(p), mean=round(p.mean()*100,2), med=round(p.median()*100,2),
                win=round((p>0).mean()*100,0), q10=round(p.quantile(0.1)*100,2),
                std=round(p.std()*100,1))
for s, hzs in (('pump',[15,30]), ('dump',[720,2880])):
    sub = X[X.stream == s]
    for hz in hzs:
        for w in ('TRAIN','VALID','TEST'):
            p = sub[(sub.win == w)][f'pnl{hz}']
            print(s, f'h={hz}', w, stats(p))
        print()

pump h=15 TRAIN {'n': 18933, 'mean': np.float64(-0.26), 'med': np.float64(-0.02), 'win': np.float64(50.0), 'q10': np.float64(-3.59), 'std': np.float64(3.3)}
pump h=15 VALID {'n': 16630, 'mean': np.float64(-0.59), 'med': np.float64(0.04), 'win': np.float64(51.0), 'q10': np.float64(-4.88), 'std': np.float64(5.4)}
pump h=15 TEST {'n': 14277, 'mean': np.float64(-0.2), 'med': np.float64(0.08), 'win': np.float64(51.0), 'q10': np.float64(-4.52), 'std': np.float64(4.5)}

pump h=30 TRAIN {'n': 18933, 'mean': np.float64(-0.2), 'med': np.float64(0.2), 'win': np.float64(53.0), 'q10': np.float64(-4.58), 'std': np.float64(4.1)}
pump h=30 VALID {'n': 16630, 'mean': np.float64(-0.66), 'med': np.float64(0.04), 'win': np.float64(50.0), 'q10': np.float64(-6.53), 'std': np.float64(6.4)}
pump h=30 TEST {'n': 14273, 'mean': np.float64(-0.17), 'med': np.float64(0.32), 'win': np.float64(54.0), 'q10': np.float64(-5.78), 'std': np.float64(5.9)}

dump h=720 TRAIN {'n': 6388, 'mean': np.float64(2.61), 'med': np.f

## 4. Дамп по кварталам: это краш 2025Q4 или смерть сигнала?

VALID/TEST по средней выглядят мёртвыми. Но per-event средняя даёт каждому
событию равный вес, а во время краша события идут тысячами одновременно — один
плохой квартал может утопить всё окно. Смотрим поквартально.

In [5]:
sub = X[X.stream == 'dump'].copy()
sub['q'] = sub.entry.dt.to_period('Q')
t = sub.groupby('q').agg(n=('k','size'),
                         m720=('pnl720', lambda s: round(s.mean()*100,2)),
                         med720=('pnl720', lambda s: round(s.median()*100,2)),
                         m30=('pnl30', lambda s: round(s.mean()*100,2)))
print('dump: по кварталам (mean/median pnl @720м, mean @30м, %)')
print(t.to_string())
pos = (t.m720 > 0).sum()
print(f'позитивных кварталов @720м: {pos} из {len(t)}')

dump: по кварталам (mean/median pnl @720м, mean @30м, %)
           n  m720  med720   m30
q                               
2024Q1  1076  5.85    7.77  2.37
2024Q2  1166  2.07    3.11 -0.87
2024Q3   598  1.96    0.91  1.46
2024Q4  1230  3.84    3.85  2.92
2025Q1  1323  2.16    1.73  1.12
2025Q2   995 -0.79   -1.22 -0.21
2025Q3  1723  0.40   -1.00  0.20
2025Q4  4685 -0.11   -1.56 -2.25
2026Q1  2932  1.88    2.01  0.99
2026Q2  4040 -0.22   -2.39 -0.05
2026Q3  1250 -1.18   -3.42 -0.67
позитивных кварталов @720м: 7 из 11


C:\Users\nvisary\AppData\Local\Temp\ipykernel_16708\4018370697.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  sub['q'] = sub.entry.dt.to_period('Q')


## Выводы nb00

**1. Книга собрана и по масштабу сходится со скаутом v2.** 70 858 событий
(49 840 pump / 21 018 dump), 585 символов с событиями. Частота событий растёт со
временем ~4x (дамп: ~1000/квартал в 2024 → 4685 в 2025Q4, 4040 в 2026Q2) —
вселенная расширяется + режим турбулентнее. Любое сравнение «раньше/позже»
обязано помнить: состав книги в конце периода другой.

**2. ⚠️ k здесь — другая величина, чем в v2.** Тут k = число триггерных БАРОВ
кластера (медиана 4, хвост 7+ = 36%). В v2 k = число ЦЕНОВЫХ ступеней заливки
(61% событий k=1, хвост 7+ = 0.6%). Числа не сопоставимы; наследованный факт
«хвост каскадов ~1%» на эту колонку не переносится. Для направления D (склейка)
понадобится отдельная колонка ценовых ступеней.

**3. Памп first-trigger шортом мёртв на всех горизонтах** (TRAIN −0.2…−2.1%,
VALID −0.6…−2.0%). Воспроизводит решающий вывод линии pump (09_3): на первом
триггере памп ещё бежит. Форма «короче = менее плохо» видна (актив №4
воспроизведён по форме). Исполнимая ценность памп-ноги, если она есть, лежит
в scale-in + классификаторе (направление B) — не в голом входе.

**4. Дамп: сильный TRAIN, затухание вне его.** TRAIN монотонно растёт с
горизонтом (+2.6% @720м, +4.2% @2880м — форма актива №4 воспроизведена), но:
7 позитивных кварталов из 11, и все слабые/отрицательные — с 2025Q2. Медиана
@2880 на VALID/TEST: −4.5% / −9.1% — типичная сделка на длинном горизонте вне
TRAIN глубоко убыточна, среднюю тянут хвостовые победители.
**Наследованный актив №1 («сигнал жив 11/11 кварталов») принадлежал
неисполнимому эталону и на честный вход НЕ переносится.**

**5. Следствие для плана.** Per-event edge честного входа вне TRAIN ≈ 0 =>
любой положительный портфельный результат (вкл. v2 «+291% / Sharpe 1.10»)
держится на том, КАКИЕ события портфель берёт (слоты, нормировка на квартал
с 4x событиями). Это усиливает приоритет направления 0 (честный портфельный
слой) и добавляет вопрос nb01: затухание — это время (те же символы стали
хуже) или состав (наплыв новых листингов)?

**Санитарные оговорки:** события перекрываются во времени (одна монета может
держать несколько позиций на длинном горизонте) — per-event средняя это
игнорирует; издержки учтены плоско (2×0.075% + слип на стопе); funding не учтён.